In [1]:
import pandas as pd
import numpy as np

# Load datasets
train_df = pd.read_csv("./data/08_Feature_Selected_Train.csv")
test_df = pd.read_csv("./data/08_Feature_Selected_Test.csv")

# Separate features and target
target = "construction_cost"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

# Dataset shapes
print("=" * 60)
print("DATASET SHAPES")
print("=" * 60)

print("Training Features:", X_train.shape)
print("Training Target:", y_train.shape)
print("Testing Features:", X_test.shape)
print("Testing Target:", y_test.shape)

# Column verification
print("\n" + "=" * 60)
print("FEATURE COLUMN VERIFICATION")
print("=" * 60)

print("Training Features:", len(X_train.columns))
print("Testing Features:", len(X_test.columns))
print("Matching Columns:", list(X_train.columns) == list(X_test.columns))

# Identify column mismatches
print("Missing in Test:", list(set(X_train.columns) - set(X_test.columns)))
print("Extra in Test:", list(set(X_test.columns) - set(X_train.columns)))

# Data types
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(X_train.dtypes.value_counts())
print("\nNon-Numeric Training Columns:")
print(X_train.select_dtypes(exclude=np.number).columns.tolist())

# Missing values
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("Training Missing Values:", X_train.isnull().sum().sum())
print("Testing Missing Values:", X_test.isnull().sum().sum())
print("Training Target Missing:", y_train.isnull().sum())
print("Testing Target Missing:", y_test.isnull().sum())

# Target distribution
print("\n" + "=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

print("Training Target:")
print(y_train.describe())

print("\nTesting Target:")
print(y_test.describe())

# Final verification
print("\n" + "=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print("All features numeric:",
      all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train.dtypes))

print("Feature columns match:",
      X_train.columns.equals(X_test.columns))

print("All values finite:",
      np.isfinite(X_train.to_numpy()).all() and
      np.isfinite(X_test.to_numpy()).all())

print("Target values finite:",
      np.isfinite(y_train.to_numpy()).all() and
      np.isfinite(y_test.to_numpy()).all())

DATASET SHAPES
Training Features: (45704, 71)
Training Target: (45704,)
Testing Features: (11426, 71)
Testing Target: (11426,)

FEATURE COLUMN VERIFICATION
Training Features: 71
Testing Features: 71
Matching Columns: True
Missing in Test: []
Extra in Test: []

DATA TYPES
int64      60
float64    11
Name: count, dtype: int64

Non-Numeric Training Columns:
[]

MISSING VALUES
Training Missing Values: 0
Testing Missing Values: 0
Training Target Missing: 0
Testing Target Missing: 0

TARGET DISTRIBUTION
Training Target:
count    4.570400e+04
mean     1.586638e+05
std      6.707041e+05
min      1.000000e+00
25%      1.000000e+04
50%      1.500000e+04
75%      3.600000e+04
max      5.000000e+07
Name: construction_cost, dtype: float64

Testing Target:
count    1.142600e+04
mean     1.565566e+05
std      6.985685e+05
min      1.000000e+00
25%      1.000000e+04
50%      1.500000e+04
75%      3.500000e+04
max      4.500000e+07
Name: construction_cost, dtype: float64

FINAL VERIFICATION
All feature

# Establish Baseline Model

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import time

# Initialize baseline model
dummy_model = DummyRegressor(strategy="mean")

# Train model
start_time = time.time()

dummy_model.fit(X_train, y_train)

training_time = time.time() - start_time

# Validation predictions
y_val_pred_dummy = dummy_model.predict(X_train)

# Evaluation
dummy_r2 = r2_score(y_train, y_val_pred_dummy)
dummy_mae = mean_absolute_error(y_train, y_val_pred_dummy)
dummy_rmse = np.sqrt(mean_squared_error(y_train, y_val_pred_dummy))

# Results
print("=" * 60)
print("DUMMY REGRESSOR - BASELINE RESULTS")
print("=" * 60)

print(f"R2 Score : {dummy_r2:.6f}")
print(f"MAE      : {dummy_mae:.2f}")
print(f"RMSE     : {dummy_rmse:.2f}")
print(f"Training Time: {training_time:.4f} seconds")

DUMMY REGRESSOR - BASELINE RESULTS
R2 Score : 0.000000
MAE      : 229854.29
RMSE     : 670696.73
Training Time: 0.0018 seconds


# Train Regression Models

In [3]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np
import time

models = {
    "Ridge": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=15, min_samples_leaf=10, random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=30, max_depth=20,
        min_samples_leaf=5, n_jobs=-1, random_state=42
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=50, max_leaf_nodes=15, random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    pred = model.predict(X_train)

    results.append({
        "Model": name,
        "R2": r2_score(y_train, pred),
        "MAE": mean_absolute_error(y_train, pred),
        "RMSE": np.sqrt(mean_squared_error(y_train, pred)),
        "Time (s)": round(elapsed, 2)
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("RMSE")

print(results_df.to_string(index=False))

                 Model       R2           MAE          RMSE  Time (s)
           Extra Trees 0.703127  51006.737064 365436.316457      1.68
Hist Gradient Boosting 0.612390  85277.052191 417564.593699      5.05
         Decision Tree 0.608383  59460.246645 419717.378591      1.20
                 Ridge 0.312931 161828.398657 555937.982256      0.07


## Validation Comparison

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

validation_results = []

for name, model in models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_val)

    validation_results.append({
        "Model": name,
        "R2": r2_score(y_val, pred),
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred))
    })

validation_df = pd.DataFrame(validation_results).sort_values("RMSE")

print(validation_df.to_string(index=False))

                 Model       R2           MAE          RMSE
           Extra Trees 0.649596  67442.328307 331321.025047
Hist Gradient Boosting 0.648265  88418.278294 331949.529607
         Decision Tree 0.598043  74697.944312 354857.909515
                 Ridge 0.381890 156781.636353 440045.302788


# Final Model Training and Test Evaluation

In [5]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import time

# Initialize final model
final_model = ExtraTreesRegressor(
    n_estimators=30,
    max_depth=20,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# Train on complete training dataset
start = time.time()
final_model.fit(X_train, y_train)
training_time = time.time() - start

# Final test predictions
y_test_pred = final_model.predict(X_test)

# Evaluation
print("=" * 50)
print("FINAL TEST RESULTS - EXTRA TREES")
print("=" * 50)

print(f"R2 Score : {r2_score(y_test, y_test_pred):.6f}")
print(f"MAE      : {mean_absolute_error(y_test, y_test_pred):.2f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")
print(f"Training Time: {training_time:.2f} seconds")

FINAL TEST RESULTS - EXTRA TREES
R2 Score : 0.541521
MAE      : 69380.23
RMSE     : 472987.65
Training Time: 1.80 seconds


# Save Final Model

In [6]:
import joblib

joblib.dump(final_model, "./data/10_Final_ExtraTrees_Model.pkl")

print("Final model saved successfully.")

Final model saved successfully.
